# Solution with parameter ambiguity aversion

<hr style="height:4px; background-color:black; border:none;">

<br>

We now consider that the planner faces ambiguity in the parameter $\gamma^i$ and $\theta^i$ that govern the land use productivity for each site $i$. The planner confronts the parameter ambiguity by performing a sensitivity analysis: minimizing the planner’s objective by altering the posterior distribution of $\varphi$ subject to a relative entropy or Kullback-Leibler penalty scaled by a parameter $\xi$.

## Model setup

To pose the robustly optimal planning problem for the Brazilian Amazon, we start by computing the intertemporal objective conditioned on the parameter vector $\varphi,$ taking into account any pure risk considerations:

```{math}
     f(d, \varphi) = \mathbb{E} \int_0^\infty e^{-\delta t}  \Bigg[ 
     P^a_t\sum_{i=1}^I A_t^i 
     -P^e \left(  \sum_{i=1}^I \kappa Z_t^i - \dot{X}_t^i \right)  
     - LC_t
     \Bigg] dt


where $\varphi:=(\gamma,\theta)$. We adopt an ex ante representation of the decision problem.  Let $\pi$ denote the baseline distribution over the parameter vector $\varphi$, constructed with computationally tractable Bayesian method.  The ambiguity-averse planner ranks alternative decision processes by solving the minimization problem:
```{math}
\min_g \int \left[ f(d, \varphi) + \xi \log g(\varphi) \right] g(\varphi) d\pi(\varphi) 



subject to $\int g(\varphi) d\pi(\varphi) = 1$.  In this formulation,
$g(\varphi) d \pi(\varphi)$ represents an altered distribution over the parameter vector $\varphi$ and $\xi \int \left[\log g(\varphi)\right] g(\varphi) d\pi(\varphi)$ penalizes departures from the baseline posterior distribution $d\pi(\varphi).$ To construct a robustly optimal allocation of land in the Brazilian Amazon over time and across space, the planner solves:
```{math}
\max_{d \in {\mathcal D}} \min_{g \ge 0, \int g d\pi = 1}   \int_{\mathcal B} f({ d},  { \varphi} )   g(\varphi) d\pi(\varphi) 
+ \xi \int_{\mathcal B} \log g(\varphi ) g({\varphi}) d\pi(\varphi) .
    




We treat parameter uncertainty probabilistically by starting with a baseline subjective prior over the parameters. We form this baseline ``prior'' conditioned on available data, as described in baseline distribution section. We construct the composite regression parameter vector, 
```{math}
    \rho' = ({\beta_\gamma}', {\nu_\gamma}', \eta_\gamma, \zeta_\gamma,    {\beta_\theta}',{\nu_\theta}', \eta_\theta, \zeta_\theta)


where the $\eta$'s are the precisions for the regression errors and the $\zeta$'s are the precisions for the random coefficients.  Uncertainty about $\rho$ induces uncertainty about the site-specific productivities, $(\gamma^i, \theta^i)$ for $i = 1,2,...,I$, which we write abstractly $
\varphi = \Phi(\rho).
$
The underlying dimension of the uncertainty is given by the sum of the number of the unknown regression parameters, the number of distinct random-effects, and the four  precision parameters that determine residual and random-effects variances for each regression equation.   We choose a parametrization such that this sum substantially is less than $2 \times I.$ This reduction turns out to be important for implementation.  With this in mind, we let ${\hat g}(\rho)$ denote the density over the underlying parameters that induces the density $g(\varphi)$ through the transformation, $\Phi(\rho),$ using a change-in-variables transformation for  densities. 

The minimization problem has a well-known quasi-analytical solution:
```{math}
\hat g^*(\rho) = \frac { \exp\left( - \frac {1} {\xi}  f\left[d, \Phi(\rho)\right] \right)} {
\int_{\mathcal R} \exp\left( - \frac {1} {\xi}  f\left[ d, \Phi(\rho)\right] \right) d \pi(\rho)} 



with a minimized objective:
```{math}
-\xi  \log \int_{\mathcal R} \exp\left( - \frac {1} {\xi}  f\left[ d, \Phi(\rho)\right] \right) d \pi(\rho).



Given the parameter ambiguity adjustment, the implied decision problem is a two-player zero-sum game.  In our problem formulation, we  may switch the  order of  maximization and minimization, which is of interest in its own right.

```{math}
\min_{\hat g \ge 0 , \int \hat g d\pi = 1} \max_{ d \in {\mathcal D}}  \int_{\mathcal R} f\left[ d,  \Phi(\rho) \right]    {\hat g}({\rho }) d\pi(\rho ) 
+ \xi \int \log {\hat g}(\rho ) {\hat g}({ \rho }) d\pi( \rho ) .


## Implementation 

### Computing ambiguity-adjusted distributions

We approximate the distributions of interest using Hamiltonian methods. These methods are typically used to compute Bayesian posteriors numerically. We extend these methods to compute ambiguity-adjusted probabilities. We would like to sample from the target distribution $ \hat g(\rho) d \overline \pi$. 

To construct the Hamiltonian simulations, we form the potential energy term $\mathcal{U}$:

```{math}
        \mathcal{U}(\rho) & = \left({\frac {1} {\xi} } \right) f\left[d, \Phi(\rho) \right] 
        - \log {\overline \pi}(\rho \mid W, R, Y, Z) \\\
        & = \left({\frac {1} {\xi} } \right) f\left[d, \Phi(\rho) \right] 
        - \log {\mathcal L}(Y|W, R, Z, \rho) - \log \pi(\rho)


so we can rewrite potential energy into objective contribution, log likelihood and log prior density (up to a constant). We treat 
$ - \frac 1 {\xi} f\left[d, \Phi(\rho)\right]$ as an additional contribution to the log-likelihood function of the parameters, even though it is mathematically distinct.


HMC relies on an auxiliary momentum vector $\omega$ of the same dimension as $\rho$, where $\omega \sim \mathcal{N}(0, M)$ and $M$ is a symmetric, positive-definite mass matrix. The Hamiltonian is then defined as:

\begin{equation}
    \mathcal{H}(\rho, \omega) :=  \mathcal{U}(\rho) + \frac{1}{2} \omega'M^{-1}\omega
\end{equation}

The HMC algorithm then consists of:
1. Initialize $\rho_{(0)}$.

2. Sample momentum $\omega_{(0)} \sim \mathcal{N}(0, M)$.

3. Generate a state proposal $(\tilde{\rho}_{(0)}, \tilde{\omega}_{(0)})$ by evolving its position according to Hamilton's equations, using the leapfrog integrator with step size $\epsilon$ and a number of steps $L$:

   $$
   \frac{d \rho}{d t} = \frac{\partial \mathcal{H}}{\partial \omega}
   $$

   $$
   \frac{d \omega}{d t} = -\frac{\partial \mathcal{H}}{\partial \rho}
   $$

4. Perform a Metropolis test to accept or reject the proposed state update $(\rho_{(1)}, \omega_{(1)}) \leftarrow (\tilde{\rho}_{(0)}, \tilde{\omega}_{(0)})$, with acceptance probability:

   $$
   \min \left\{ 1,\ \exp\left( \mathcal{H}(\rho_{(0)}, \omega_{(0)}) - \mathcal{H}(\tilde{\rho}_{(0)}, \tilde{\omega}_{(0)}) \right) \right\}
   $$

5. Repeat steps 2–4 to generate a total of 4000 samples by running HMC simultaneously across 4 independent chains, each producing 1000 samples with 500 burn-in samples per chain.


We use Stan software for the computation. Below is the $\bf{Stan}$ code for adjusted distribution sampling. 

To save computation time, we do not compute the whole components of $f\left[d, \Phi(\rho) \right]$. Note that the leapfrog process only depends on components related to $\rho$. Therefore, we only need to compute $\mathbb{E} \int_0^\infty e^{-\delta t}  \Bigg[ 
     P^a_t\sum_{i=1}^I A_t^i 
     -P^e \left(  \sum_{i=1}^I  - \dot{X}_t^i \right)  
     \Bigg] dt$


```{code-block} python
### adjusted.stan ###

functions {
  real log_value(vector gamma, vector theta, int T, int S, matrix Z,
                 vector forest_area_2017, vector alpha_p_Adym, matrix Bdym,
                 vector ds_vect, real xi, real pa, real pe,
                 matrix carbon_stock) {
    // Compute stock of carbon (X)
    row_vector[T] omega = gamma' * carbon_stock;
    vector[T + 1] X;
    X[1] = gamma' * forest_area_2017;
    X[2 : (T + 1)] = alpha_p_Adym * X[1] + Bdym * omega';

    // Compute aggregate X dot
    vector[T] Xdot_agg = (X[2 : (T + 1)] - X[1 : T]);
    real term_1 = -pe * sum(ds_vect .* (-Xdot_agg));

    // Value of agricultural output
    vector[T + 1] agri_output = pa * (theta' * Z)';
    real term_2 = sum(ds_vect .* agri_output[2 : T + 1]);

    // Overall objective value
    real obj_val = term_1 + term_2;
    real log_density_val = -1.0 / xi * obj_val;

    return log_density_val;
  }
}
data {
  // Planner problem
  int<lower=0> T; // Time horizon
  int<lower=0> S; // Number of sites
  matrix[S, T + 1] Z; // Agricultural area state
  matrix[S, T] U; // U control
  matrix[S, T] V; // V control
  vector[S] zbar_2017; // z_bar in 2017
  vector[S] forest_area_2017; // forest area in 2017
  vector[T] alpha_p_Adym;
  matrix[T, T] Bdym;
  vector[T] ds_vect; // Time discounting vector
  matrix[S, T] carbon_stock;
  real<lower=0> alpha; // Mean-reversion coefficient
  real zeta_u; // Penalty on adjustment costs
  real zeta_v; // Penalty on adjustment costs
  real xi; // Penalty on prior-posterior KL div
  real kappa; // Effect of cattle farming on emissions
  real<lower=0> pa; // Price of cattle output
  real<lower=0> pe; // Price of carbon emission transfers

  // Gamma regression
  int<lower=1> N_gamma; // total number of observations
  int<lower=1> K_gamma; // number of predictors
  int<lower=1> M_gamma; // number of groups
  matrix[N_gamma, K_gamma] X_gamma; // design matrix
  vector[N_gamma] y_gamma; // outcome variable
  array[N_gamma] int m_gamma; // group map

  // Theta regression
  int<lower=1> N_theta; // total number of observations
  int<lower=1> K_theta; // number of predictors
  int<lower=1> M_theta; // number of groups
  matrix[N_theta, K_theta] X_theta; // design matrix
  vector[N_theta] y_theta; // outcome variable
  array[N_theta] int m_theta; // group map
  vector[N_theta] W_theta; // weights

  // Gamma projection
  int<lower=1> num_sites; // number of samples
  matrix[num_sites, K_gamma] X_gamma_fit; // design matrix
  array[num_sites] int m_gamma_fit; // group map

  // Theta projection
  int<lower=1> C_theta_fit; // Number of municipalities
  matrix[C_theta_fit, K_theta] X_theta_fit; // design matrix
  array[C_theta_fit] int m_theta_fit; // group map

  // Sparse G_theta
  int<lower=0> G_nnz_theta; // Number of non-zero elements
  vector[G_nnz_theta] G_w_theta; // Non-zero values
  array[G_nnz_theta] int G_v_theta; // Column indices (1-based in Stan)
  array[num_sites + 1] int G_u_theta; // Row pointers (1-based in Stan)
  real pa_2017; // price of cattle in 2017
}
transformed data {
  vector[N_theta] y_theta_w = W_theta .* y_theta;
  matrix[N_theta, K_theta] X_theta_w;
  for (n in 1 : N_theta)
    X_theta_w[n] = W_theta[n] * X_theta[n];
}
parameters {
  // Gamma regression parameters
  vector[K_gamma] beta_gamma;
  vector[M_gamma] nu_gamma_transform;
  real log_precision_u_gamma;
  real log_precision_v_gamma;

  // Theta regression parameters
  vector[K_theta] beta_theta;
  vector[M_theta] nu_theta_transform;
  real log_precision_u_theta;
  real log_precision_v_theta;
  // real<lower=log(1e-5), upper=log(1e5)> log_precision_v_theta;
}
transformed parameters {
  real sigma_u_gamma = exp(-0.5 * log_precision_u_gamma);
  real sigma_v_gamma = exp(-0.5 * log_precision_v_gamma);
  real sigma_u_theta = exp(-0.5 * log_precision_u_theta);
  real sigma_v_theta = exp(-0.5 * log_precision_v_theta);

  vector[M_gamma] nu_gamma = sigma_v_gamma * nu_gamma_transform;
  vector[M_theta] nu_theta = sigma_v_theta * nu_theta_transform;

  // Pre-multiply theta FE's by weights
  vector[N_theta] nu_theta_w = W_theta .* nu_theta[m_theta];
  vector[C_theta_fit] nu_theta_fit = nu_theta[m_theta_fit];
  vector[N_gamma] nu_gamma_sort = nu_gamma[m_gamma];
  vector[num_sites] nu_gamma_fit = nu_gamma[m_gamma_fit];

  // Projection
  vector<lower=0>[num_sites] gamma = exp(X_gamma_fit * beta_gamma
                                         + nu_gamma_fit);

  vector[C_theta_fit] exp_log_theta = exp(X_theta_fit * beta_theta
                                          + nu_theta_fit);
  vector<lower=0>[num_sites] theta = csr_matrix_times_vector(num_sites,
                                                             C_theta_fit,
                                                             G_w_theta,
                                                             G_v_theta,
                                                             G_u_theta,
                                                             exp_log_theta)
                                     / pa_2017;
}
model {

  nu_gamma_transform ~ normal(0, 1);
  nu_theta_transform ~ normal(0, 1);

  y_gamma ~ normal(X_gamma * beta_gamma + nu_gamma_sort, sigma_u_gamma);
  y_theta_w ~ normal(X_theta_w * beta_theta + nu_theta_w, sigma_u_theta);

  target += log_value(gamma, theta, T, S, Z, forest_area_2017, alpha_p_Adym,
                      Bdym, ds_vect, xi, pa, pe, carbon_stock);
}


### Iteratively solve the maxi-min problem


Notice that these computations take as given a contingent decision theory sequence. To address this, we iteratly compute action $d$ by maximizing and an ambiguity-adjusted distribution over unknown productivity parameters by minimizing subject to penalization. 

What matters for the maximization step is the mean of the productivity parameters under the ambiguity-adjusted distribution for $\rho.$ Let ${\bar \varphi}^*$ denote this mean as computed using the robustly optimal decision, $d^*.$

To find the robustly optimal, we initialize the algorithm by setting ${\bar \varphi}_{(0)}$ as the mean of $\varphi$ implied by the baseline distribution $\pi$ over $\rho$ and setting $d_{(0)}$ as the maximized solution taking this choice of ${\bar \varphi}_{(0)}$ as given.  We then iterate as follows;

1. Given ${\bar \varphi_{s}}$, solve the planner's problem for the decision vector $d_{(s)}$.

2. Given $d_{(s)}$, compute the ambiguity-adjusted distribution over $\rho$, and calculate the mean ${\tilde \varphi}_{s}$ of the implied ambiguity-adjusted distribution over $\varphi$. Then update:

   $$
   {\bar \varphi}_{s+1} = 0.75\, {\bar \varphi}_{s} + 0.25\, {\tilde \varphi}_{s}
   $$

3. If $\left\| {\bar \varphi}_{s+1} - \varphi_{s} \right\|_{\infty} < 0.005$, stop. Otherwise, return to step 1 using the updated ${\bar \varphi}_{s+1}$.



In the code below we iteratively solve the planner problem until convergence and save all the convergent distributions:

```{code-block} python

def sample(
    xi,
    pe,
    pa,
    weight,
    num_sites,
    # Model parameters
    T,
    alpha=0.045007414,
    delta=0.02,
    kappa=2.094215255,
    zeta_u=1.66e-4 * 1e9,
    zeta_v=1.00e-4 * 1e9,
    norm_fac=1e9,
    pa_2017=44.9736197781184,
    # Optimizer
    solver="gurobi",
    # Sampling params
    max_iter=20000,
    tol=0.005,
    final_sample_size=2_000,
    **stan_kwargs,
):
    
    
    baseline_distribution = pd.read_csv(get_path("data","calibration")/f'distribution_parameters_all_{num_sites}.csv')
    beta_gamma_cols = [col for col in baseline_distribution.columns if col.startswith('beta_gamma')]
    beta_theta_cols = [col for col in baseline_distribution.columns if col.startswith('beta_theta')]
    nu_gamma_cols = [col for col in baseline_distribution.columns if col.startswith('nu_gamma')]
    nu_theta_cols = [col for col in baseline_distribution.columns if col.startswith('nu_theta')]
    

    inits = {
        "log_precision_u_gamma": np.log(1/(baseline_distribution["sigma_u_gamma"].mean())),
        "log_precision_v_gamma": np.log(1/(baseline_distribution["sigma_v_gamma"].mean())),
        "log_precision_u_theta": np.log(1/(baseline_distribution["sigma_u_theta"].mean())),
        "log_precision_v_theta": np.log(1/(baseline_distribution["sigma_v_theta"].mean())),
        "beta_gamma": baseline_distribution[beta_gamma_cols].mean().tolist(),
        "beta_theta": baseline_distribution[beta_theta_cols].mean().tolist(),
        "nu_gamma_transform": [0.01] * len(nu_gamma_cols),
        "nu_theta_transform": [0.01] * len(nu_theta_cols),
    }

    stan_kwargs['inits']=inits
    

    pickle_file = 'stan_model/compiled_model.pkl'

    if os.path.exists(pickle_file):
        # Load the model from the pickle file
        sampler = pickle.load(open(pickle_file, 'rb'))
        print("Loaded model from pickle.")
    else:
        # Compile the Stan model and save it to the pickle file
        sampler = CmdStanModel(
            stan_file=get_path("stan_model") / "adjusted.stan",
            cpp_options={"STAN_THREADS": "true"},
            force_compile=True,
        )
        with open(pickle_file, 'wb') as f:
            pickle.dump(sampler, f)
        print("Compiled model and saved to pickle.")


    # Load site data
    (zbar_2017, z_2017, forest_area_2017) = load_site_data(num_sites,norm_fac=norm_fac)
    # print("zbar_2017", zbar_2017)
    # print("z_2017", z_2017)
    # print("forest_area_2017", forest_area_2017)

    # Load parameter regression data
    # (site_theta_df, site_gamma_df) = load_reg_data(num_sites)

    # Set initial theta & gamma using baseline mean
    (theta_vals, gamma_vals) = load_productivity_params(num_sites)

    # Save starting params
    uncertain_vals = np.concatenate((theta_vals, gamma_vals)).copy()
    uncertain_vals_old = np.concatenate((theta_vals, gamma_vals)).copy()

    # Collected Ensembles over all iterations; dictionary indexed by iteration number
    collected_ensembles = {}
    coe_ensembles = {}

    # Track error over iterations
    uncertain_vals_tracker = [uncertain_vals_old.copy()]
    abs_error_tracker = []
    pct_error_tracker = []
    solution_tracker = []
    sampling_time_tracker = []
    fit_tracker = []

    # Results dictionary
    results = dict(
        num_sites=num_sites,
        tol=tol,
        T=T,
        delta_t=delta,
        alpha=alpha,
        kappa=kappa,
        pf=pe,
        pa=pa,
        xi=xi,
        final_sample_size=final_sample_size,
        weight=weight,
    )

    # Initialize error & iteration counter
    abs_error = np.infty
    pct_error = np.infty
    cntr = 0

    # Loop until convergence
    while cntr < max_iter and pct_error > tol:
        print(f"Optimization Iteration[{cntr+1}/{max_iter}]\n")
        
        if cntr > 0:
        
            inits = {
            "log_precision_u_gamma": fit.stan_variable("log_precision_u_gamma").mean(),
            "log_precision_v_gamma": fit.stan_variable("log_precision_v_gamma").mean(),
            "log_precision_u_theta": fit.stan_variable("log_precision_u_theta").mean(),
            "log_precision_v_theta": fit.stan_variable("log_precision_v_theta").mean(),
            "beta_gamma": fit.stan_variable("beta_gamma").mean(axis=0),
            "beta_theta": fit.stan_variable("beta_theta").mean(axis=0),
            "nu_gamma_transform": [0.1] * len(nu_gamma_cols),
            "nu_theta_transform": [0.1] * len(nu_theta_cols),
            }

            stan_kwargs['inits']=inits
        

        # Flatten uncertain values
        uncertain_vals = np.asarray(uncertain_vals).flatten()

        # Unpacking uncertain values
        theta_vals = uncertain_vals[:num_sites].copy()
        gamma_vals = uncertain_vals[num_sites:].copy()

        print(f"Theta: {theta_vals}\n")
        print(f"Gamma: {gamma_vals}\n")

        # Computing carbon absorbed in start period
        x0_vals = gamma_vals * forest_area_2017

        # Solve planner problem
        planner_solution = solve_planner_problem(
            time_horizon=T,
            theta=theta_vals,
            gamma=gamma_vals,
            x0=x0_vals,
            z0=z_2017,
            zbar=zbar_2017,
            price_emissions=pe,
            price_cattle=pa,
            alpha=alpha,
            delta=delta,
            kappa=kappa,
            zeta_u=zeta_u,
            zeta_v=zeta_v,
            solver=solver,
        )

        # Update trackers
        solution_tracker.append(planner_solution)

        # HMC sampling
        print("Starting HMC sampling...\n")
        model_data = dict(
            T=T,
            S=num_sites,
            alpha=alpha,
            zbar_2017=zbar_2017,
            forest_area_2017=forest_area_2017,
            zeta_u=zeta_u,
            zeta_v=zeta_v,
            xi=xi,
            kappa=kappa,
            pa=pa,
            # pa_2017=pa_2017,
            pe=pe,
            num_sites=num_sites,
            **vectorize_trajectories(planner_solution),
            **_dynamics_matrices(T, alpha, delta),
            **_precompute_decision(planner_solution, zbar_2017, alpha),
            **load_gamma_calib(num_sites, "reg"),
            **load_gamma_calib(num_sites, "fit"),
            **load_theta_calib(num_sites, "reg"),
            **load_theta_calib(num_sites, "fit"),
        )


        # Sampling from adjusted distribution
        sampling_time = time.time()
        fit = sampler.sample(
            data=model_data,
            **stan_kwargs,
        )
        sampling_time = time.time() - sampling_time
        print(f"Finished sampling! Elapsed Time: {sampling_time} seconds\n")
        print(fit.diagnose())

        # Update fit and sampling time trackers
        fit_tracker.append(fit.summary())
        sampling_time_tracker.append(sampling_time)

        # Extract samples
        theta_adj_samples = fit.stan_variable("theta")
        gamma_adj_samples = fit.stan_variable("gamma")
        theta_coe_adj_samples = fit.stan_variable("beta_theta")
        gamma_coe_adj_samples = fit.stan_variable("beta_gamma")
        theta_nu_adj_samples = fit.stan_variable("nu_theta")
        gamma_nu_adj_samples = fit.stan_variable("nu_gamma")
        theta_sigma_u_adj_samples = fit.stan_variable("sigma_u_theta")[:, None]
        gamma_sigma_u_adj_samples = fit.stan_variable("sigma_u_gamma")[:, None]
        theta_sigma_v_adj_samples = fit.stan_variable("sigma_v_theta")[:, None]
        gamma_sigma_v_adj_samples = fit.stan_variable("sigma_v_gamma")[:, None]

        uncertainty_adj_samples = np.concatenate(
            (theta_adj_samples, gamma_adj_samples), axis=1
        )

        uncertainty_coe_adj_samples = np.concatenate(
            (theta_coe_adj_samples, gamma_coe_adj_samples,theta_nu_adj_samples,gamma_nu_adj_samples,theta_sigma_u_adj_samples,gamma_sigma_u_adj_samples,theta_sigma_v_adj_samples,gamma_sigma_v_adj_samples), axis=1
        )

        # Update ensemble/tracker
        collected_ensembles.update({cntr: uncertainty_adj_samples.copy()})
        coe_ensembles.update({cntr: uncertainty_coe_adj_samples.copy()})

        print(f"Parameters from last iteration: {uncertain_vals_old}\n")
        print(
            f"""Parameters from current iteration:
            {np.mean(uncertainty_adj_samples, axis=0)}\n"""
        )

        # Compute exponentially-smoothened new params
        uncertain_vals = (
            weight * np.mean(uncertainty_adj_samples, axis=0)
            + (1 - weight) * uncertain_vals_old
        )

        uncertain_vals_tracker.append(uncertain_vals.copy())
        print(f"Updated uncertain values: {uncertain_vals}\n")

        # Evaluate error for convergence check
        # The percentage difference are changed to absolute difference
        abs_error = np.max(np.abs(uncertain_vals_old - uncertain_vals))
        pct_error = np.max(
            np.abs(uncertain_vals_old - uncertain_vals) / uncertain_vals_old
        )

        abs_error_tracker.append(abs_error)
        pct_error_tracker.append(pct_error)

        print(
            f"""
            Iteration [{cntr+1:4d}]: Absolute Error = {abs_error},
            Percentage Error = {pct_error}
            """
        )

        # Exchange parameter values
        uncertain_vals_old = uncertain_vals

        # Increase the counter
        cntr += 1

        # Update results directory
        results.update(
            {
                "cntr": cntr,
                "abs_error_tracker": np.asarray(abs_error_tracker),
                "pct_error_tracker": np.asarray(pct_error_tracker),
                "uncertain_vals_tracker": np.asarray(uncertain_vals_tracker),
                "sampling_time_tracker": sampling_time_tracker,
                "collected_ensembles": collected_ensembles,
                "solution_tracker": solution_tracker,
                "coe_ensembles": coe_ensembles,
                "fit_tracker": fit_tracker,
            }
        )

    # Sample (densly) the final distribution
    print("Terminated. Sampling the final distribution...\n")
    stan_kwargs["iter_sampling"] = final_sample_size
    fit = sampler.sample(
        data=model_data,
        **stan_kwargs,
    )

    # Extract samples
    theta_adj_samples = fit.stan_variable("theta")
    gamma_adj_samples = fit.stan_variable("gamma")
    theta_coe_adj_samples = fit.stan_variable("beta_theta")
    gamma_coe_adj_samples = fit.stan_variable("beta_gamma")
    theta_nu_adj_samples = fit.stan_variable("nu_theta")
    gamma_nu_adj_samples = fit.stan_variable("nu_gamma")
    theta_sigma_u_adj_samples = fit.stan_variable("sigma_u_theta")[:, None]
    gamma_sigma_u_adj_samples = fit.stan_variable("sigma_u_gamma")[:, None]
    theta_sigma_v_adj_samples = fit.stan_variable("sigma_v_theta")[:, None]
    gamma_sigma_v_adj_samples = fit.stan_variable("sigma_v_gamma")[:, None]

    final_samples = np.concatenate((theta_adj_samples, gamma_adj_samples), axis=1)
    final_samples_coe = np.concatenate(
        (theta_coe_adj_samples, gamma_coe_adj_samples,theta_nu_adj_samples,gamma_nu_adj_samples,theta_sigma_u_adj_samples,gamma_sigma_u_adj_samples,theta_sigma_v_adj_samples,gamma_sigma_v_adj_samples), axis=1
    )

    results.update({"final_sample": final_samples})
    results.update({"final_sample_coe": final_samples_coe})

    # results.update({"eta_sample": eta_samples})
    # results.update({"nu_sample": nu_samples})
    # results.update({"V_gamma_sample": V_gamma_samples})
    # results.update({"V_theta_sample": V_theta_samples})

    return results


def _dynamics_matrices(T, alpha, delta, dt=1):
    # Create dynamics matrices
    arr = np.cumsum(
        np.triu(np.ones((T, T))),
        axis=1,
    ).T
    Bdym = (1 - alpha) ** (arr - 1)
    Bdym[Bdym > 1] = 0.0
    Adym = np.arange(1, T + 1)
    alpha_p_Adym = np.power(1 - alpha, Adym)

    # Other placeholders!
    ds_vect = np.exp(-delta * np.arange(T) * dt)
    ds_vect = np.reshape(ds_vect, (ds_vect.size, 1)).flatten()
    return {"alpha_p_Adym": alpha_p_Adym, "Bdym": Bdym, "ds_vect": ds_vect}


def _precompute_decision(PlannerSolution,zbar,alpha):
    Z = PlannerSolution.Z.T
    U = PlannerSolution.U[:-1, :].T
    forest_area = zbar.reshape(-1, 1) - Z[:,:-1]
    carbon_stock=(alpha * forest_area) - U
    
    return {"carbon_stock": carbon_stock}

## Adjusted distribution analysis

### Shadow price
First we similarly compute the implied shadow price under ambiguity adjusted distributions.

```{code-block} python

(
    zbar_1995,
    z_1995,
    forest_area_1995,
    z_2008,
    theta,
    gamma,
) = load_site_data_1995(sitenum)

pe_values = np.arange(pe_low, pe_high, 0.1)
results = []
for pe in pe_values:
    samples = adjusted.sample(
        xi=xi,
        pe=pe,
        pa=pa,
        weight=0.25,
        num_sites=sitenum,
        T=200,
        solver=solver,
        max_iter=100,
        final_sample_size=4_000,
        iter_sampling=4000,
        iter_warmup=500,
        show_progress=True,
        seed=1,
        # inits=0.1,
    )

    theta = np.mean(samples["final_sample"][:, :sitenum], axis=0)
    gamma = np.mean(samples["final_sample"][:, sitenum:], axis=0)
    result = shadow_price_opt(
        zbar_1995,
        z_1995,
        forest_area_1995,
        z_2008,
        theta,
        gamma,
        sitenum=sitenum,
        solver=solver,
        timehzn=200,
        pa=pa,
        pe=pe,
    )
    results.append(result)
results = np.array(results)

min_index = np.argmin(results)
min_result = results[min_index]
min_pe = pe_values[min_index]


def shadow_price_opt(
    zbar_1995,
    z_1995,
    forest_area_1995,
    z_2008,
    theta,
    gamma,
    sitenum=78,
    solver="gurobi",
    timehzn=200,
    pa=41.11,
    pe=7.1,
    model="det",
):
    
    pa_list = load_price_data()
    price_cattle = np.concatenate((pa_list, np.full(200 - len(pa_list), pa)))
    
    # Computing carbon absorbed in start period
    x0_vals_1995 = gamma * forest_area_1995

    # if model == "mpc":
    #     solve_planner_problem = gams.mpc_shadow_price

    results = solve_planner_problem(
        theta=theta,
        gamma=gamma,
        x0=x0_vals_1995,
        zbar=zbar_1995,
        z0=z_1995,
        price_emissions=pe,
        price_cattle=price_cattle,
        solver=solver,
    )
    Z = results.Z
    z_2008_agg = np.sum(z_2008) / 1e9
    ratio = (np.sum(Z[13]) - z_2008_agg) / z_2008_agg

    return ratio

```{table}  Business-as-usual prices


| number of sites | agricultural price | $\xi$    | carbon price ($P^{ee}$) |
|-----------------|--------------------|----------|-------------------------|
| 1043            | $p^a = 41.1$       | $\infty$ | 6.6                     |
| 1043            | $p^a = 41.1$       | 2        | 5.5                     |
| 1043            | $p^a = 41.1$       | 1        | 4.7                     |
| 1043            | $p^a = 41.1$       | 0.5      | 2.9                     |
```


### Densities

The ambiguity adjustments to the productivity parameter distributions are very heterogeneous across all sites and very different under different transfers $b$. We can compute the relative entropy:
$
D_{\mathrm{KL}}(P \,\|\, Q) = \int P(x) \log \left( \frac{P(x)}{Q(x)} \right) dx
$

for each site and each level of $b$.

In the code below we compute the relative entropy for $\theta$ and $\gamma$ separately under business as usual price and $b=15$.

```{code-block} python

import pandas as pd
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
from scipy.stats import gaussian_kde
from scipy.stats import entropy
import geopandas as gpd
import pandas as pd
from tqdm import tqdm
from pysrc.analysis.figures import land_allocation,density,trajectory_diff
from matplotlib.backends.backend_pdf import PdfPages
import sys
from pysrc.services.file_service import get_path

import argparse
parser = argparse.ArgumentParser(description="shadow price calculation")
parser.add_argument("--pee",type=float,default=4.7)
parser.add_argument("--xi",type=float,default=1)
parser.add_argument("--sites",type=int,default=1043)

args = parser.parse_args()
pee=args.pee
xi=args.xi
num_sites=args.sites

solver="gurobi"
pa=41.11
pee_det=6.6


result_folder = os.path.join(
    str(get_path("output")),
    "sampling",
    solver,
    f"{num_sites}sites",
    f"pa_{pa}",
    f"xi_{xi}",
)
prior_folder = os.path.join(
    str(get_path("output")),
    "sampling",
    solver,
    f"{num_sites}sites",
    f"pa_{pa}",
    "xi_10000.0",
)

with open(result_folder + f"/pe_{pee}/results.pcl", "rb") as f:
    b0 = pickle.load(f)

with open(result_folder + f"/pe_{pee+15}/results.pcl", "rb") as f:
    b15 = pickle.load(f)

with open(prior_folder + f"/pe_{pee_det+15}/results.pcl", "rb") as f:
    results_unadjusted = pickle.load(f)
    
    
    
theta_unadjusted = results_unadjusted["final_sample"][:16000, :num_sites]
gamma_unadjusted = results_unadjusted["final_sample"][:16000, num_sites:]
theta_adjusted_b0 = b0["final_sample"][:16000, :num_sites]
gamma_adjusted_b0 = b0["final_sample"][:16000, num_sites:]
theta_adjusted_b15 = b15["final_sample"][:16000, :num_sites]
gamma_adjusted_b15 = b15["final_sample"][:16000, num_sites:]




def compute_kl(unadj, adj):
    common_grid = np.linspace(min(unadj.min(), adj.min()), max(unadj.max(), adj.max()), 100)
    p = gaussian_kde(unadj, bw_method='scott')(common_grid) + 1e-20
    q = gaussian_kde(adj, bw_method='scott')(common_grid) + 1e-20
    return entropy(p, q)

kl_data = {"id": np.arange(1, num_sites + 1)}

print("Computing theta KL divergences...")
for label, theta_hmc in zip(['theta_b0', 'theta_b15'], [theta_adjusted_b0, theta_adjusted_b15]):
    kl_data[label] = [
        compute_kl(theta_unadjusted[:, i], theta_hmc[:, i])
        for i in tqdm(range(num_sites), desc=label)
    ]

print("Computing gamma KL divergences...")
for label, gamma_hmc in zip(['gamma_b0', 'gamma_b15'], [gamma_adjusted_b0, gamma_adjusted_b15]):
    kl_data[label] = [
        compute_kl(gamma_unadjusted[:, i], gamma_hmc[:, i])
        for i in tqdm(range(num_sites), desc=label)
    ]

kl_df = pd.DataFrame(kl_data)

output_folder = str(get_path("output")) + f"/figures/entropy/site_{num_sites}/xi{xi}/"
os.makedirs(output_folder, exist_ok=True)
kl_df.to_csv(output_folder + 'kl_divergences_theta_gamma.csv', index=False)

# ------------------------
# Top-2 divergent sites per column
# ------------------------
top_kl_divergences = {}
for col in ['theta_b0', 'theta_b15', 'gamma_b0', 'gamma_b15']:
    top_sites = kl_df.nlargest(2, col)
    top_kl_divergences[col] = top_sites[['id', col]].values.tolist()

print("Top 2 KL divergences per parameter:")
for param, top in top_kl_divergences.items():
    for site, value in top:
        print(f"{param}: id {int(site)} → KL = {value:.4f}")
        
        
        
print("start plot densities")

density(num_sites=num_sites,pee=pee,xi=xi,solver=solver)


Below we plot the ranking of relative entropy from small to large values, over 1043 sites. (1st is the lowest and 1043th is the highest)

In [ ]:
import plotly.graph_objects as go
import base64

# Load and encode the two images
with open("../data/figures/re_theta_b0.png", "rb") as f:
    img_b0 = base64.b64encode(f.read()).decode()

with open("../data/figures/re_theta_b15.png", "rb") as f:
    img_b15 = base64.b64encode(f.read()).decode()

# Construct figure
fig = go.Figure()

# Add both images (index 0 = b=0, index 1 = b=15)
fig.add_layout_image(
    dict(
        source="data:image/png;base64," + img_b0,
        xref="x", yref="y",
        x=0, y=1, sizex=1, sizey=1,
        layer="below",
        opacity=1,
    )
)

fig.add_layout_image(
    dict(
        source="data:image/png;base64," + img_b15,
        xref="x", yref="y",
        x=0, y=1, sizex=1, sizey=1,
        layer="below",
        opacity=0,
    )
)

# Add a dummy trace
fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers", marker_opacity=0))

# Initial annotation (LaTeX-style title with theta)
fig.update_layout(
    annotations=[
        dict(
            text="ranking of relative entropy for θ, b = 0",
            x=0.5, y=1.05,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=18),
            align="center"
        )
    ]
)

# Buttons to toggle image opacity and update title
fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            buttons=[
                dict(
                    label="b = 0",
                    method="relayout",
                    args=[{
                        "images[0].opacity": 1,
                        "images[1].opacity": 0,
                        "annotations": [
                            dict(
                                text="ranking of relative entropy for θ, b = 0",
                                x=0.5, y=1.05,
                                xref="paper", yref="paper",
                                showarrow=False,
                                font=dict(size=18),
                                align="center"
                            )
                        ]
                    }]
                ),
                dict(
                    label="b = 15",
                    method="relayout",
                    args=[{
                        "images[0].opacity": 0,
                        "images[1].opacity": 1,
                        "annotations": [
                            dict(
                                text="ranking of relative entropy for θ, b = 15",
                                x=0.5, y=1.05,
                                xref="paper", yref="paper",
                                showarrow=False,
                                font=dict(size=18),
                                align="center"
                            )
                        ]
                    }]
                ),
            ],
            x=0.1,
            y=1.15,
            showactive=True
        )
    ],
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(visible=False, range=[0, 1]),
    margin=dict(t=30, b=135),
    height=600
)

fig.show()


In [ ]:
import plotly.graph_objects as go
import base64

# Load and encode the two images
with open("../data/figures/re_gamma_b0.png", "rb") as f:
    img_b0 = base64.b64encode(f.read()).decode()

with open("../data/figures/re_gamma_b15.png", "rb") as f:
    img_b15 = base64.b64encode(f.read()).decode()

# Construct figure
fig = go.Figure()

# Add both images (index 0 = b=0, index 1 = b=15)
fig.add_layout_image(
    dict(
        source="data:image/png;base64," + img_b0,
        xref="x", yref="y",
        x=0, y=1, sizex=1, sizey=1,
        layer="below",
        opacity=1,
    )
)

fig.add_layout_image(
    dict(
        source="data:image/png;base64," + img_b15,
        xref="x", yref="y",
        x=0, y=1, sizex=1, sizey=1,
        layer="below",
        opacity=0,
    )
)

# Add a dummy trace
fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers", marker_opacity=0))

# Initial annotation (LaTeX-style title with gamma)
fig.update_layout(
    annotations=[
        dict(
            text="ranking of relative entropy for γ, b = 0",
            x=0.5, y=1.05,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=18),
            align="center"
        )
    ]
)

# Buttons to toggle image opacity and update title
fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            buttons=[
                dict(
                    label="b = 0",
                    method="relayout",
                    args=[{
                        "images[0].opacity": 1,
                        "images[1].opacity": 0,
                        "annotations": [
                            dict(
                                text="ranking of relative entropy for γ, b = 0",
                                x=0.5, y=1.05,
                                xref="paper", yref="paper",
                                showarrow=False,
                                font=dict(size=18),
                                align="center"
                            )
                        ]
                    }]
                ),
                dict(
                    label="b = 15",
                    method="relayout",
                    args=[{
                        "images[0].opacity": 0,
                        "images[1].opacity": 1,
                        "annotations": [
                            dict(
                                text="ranking of relative entropy for γ, b = 15",
                                x=0.5, y=1.05,
                                xref="paper", yref="paper",
                                showarrow=False,
                                font=dict(size=18),
                                align="center"
                            )
                        ]
                    }]
                ),
            ],
            x=0.1,
            y=1.15,
            showactive=True
        )
    ],
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(visible=False, range=[0, 1]),
    margin=dict(t=30, b=135),
    height=600
)

fig.show()


We then plot the baseline and ambiguity-adjusted densities for $\theta$ and $\gamma$. The default site is the site with the highest relative entropy. All densities of 1043 sites are presented below.  

In [ ]:

import pandas as pd
import os
import plotly.figure_factory as ff
import numpy as np
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
from matplotlib.backends.backend_pdf import PdfPages
root_folder='/project/lhansen/HMC_book/Amazon/docs'
os.chdir(root_folder)

# import pathlib
# temp = pathlib.PosixPath
# pathlib.PosixPath = pathlib.WindowsPath


with open("data/b0.pcl", 'rb') as f:
    # Load the data from the file
    b0 = pickle.load(f)
with open("data/b15.pcl", 'rb') as f:
    # Load the data from the file
    b15 = pickle.load(f)

with open('data/prior.pcl', 'rb') as f:
    # Load the data from the file
    results_unadjusted = pickle.load(f)
    
site=1043

# pdf_pages = PdfPages('mixture.pdf')
Num=4000
theta_unadjusted=results_unadjusted['theta']
gamma_unadjusted=results_unadjusted['gamma']
theta_adjusted_b0=b0['final_sample'][:Num,:site]
gamma_adjusted_b0=b0['final_sample'][:Num,site:]
theta_adjusted_b15=b15['final_sample'][:Num,:site]
gamma_adjusted_b15=b15['final_sample'][:Num,site:]

initial_site_index = 903 

sites = site  # Total number of sites, replace `site` with actual variable if not defined

# Creating the initial figure that will be updated
fig = go.Figure()

# Loop to add all KDE plots to the figure but set them to be invisible initially
for idx in range(sites):
    data = [gamma_unadjusted[:, idx], gamma_adjusted_b0[:, idx], gamma_adjusted_b15[:, idx]]
    group_labels = ['Baseline', 'b=0', 'b=15']
    colors = ['black', 'red', 'blue']
    
    # We create one distplot for each site and store them in different traces
    distplot = ff.create_distplot(data, group_labels, show_hist=False, show_rug=False, colors=colors,bin_size=0.5)
    trace_idx = 0
    for trace in distplot.data:
        trace.visible = False  # Set all to invisible initially
        if trace_idx == 1 or trace_idx == 2:
            trace.update(mode='lines', fill='tozeroy')
        trace.update(line=dict(width=4))
        fig.add_trace(trace)
        trace_idx += 1

# Creating buttons for the dropdown menu
buttons = []
initial_visibility = [False] * len(fig.data) 
for idx in range(sites):
    visibility = [False] * len(fig.data)
    
    # We need to toggle visibility of traces in groups of 3 (as each distplot added 3 traces)
    for j in range(idx * 3, (idx + 1) * 3):
        visibility[j] = True
    
    button = dict(
        label=f'Site {idx + 1}',
        method='update',
        args=[{'visible': visibility},
              {'title': f'Figure 9(a) Probability Density for γ at Site {idx + 1}'}]
    )
    buttons.append(button)
    if idx == initial_site_index:
        initial_visibility = visibility.copy()
        
# Update initial visibility 
for i, vis in enumerate(initial_visibility):
    fig.data[i].visible = vis
    
# Adding the dropdown menus to the figure
fig.update_layout(
    updatemenus=[{
        'buttons': buttons,
        'active': initial_site_index,
        'direction': 'down',
        'showactive': True,
        'x': 0.8,
        'xanchor': 'center',
        'y': 1.15,
        'yanchor': 'top'
    }],
    title=f'Probability Density for γ at Site {initial_site_index + 1}',
    xaxis_title='Parameter Value',
    yaxis_title='Density',
    width=800,  # Width of the figure in pixels
    height=600  # Height of the figure in pixels
)

for tr in fig.data:
    tr.x = tr.x[::5]        # 300 pts → 60 pts
    tr.y = tr.y[::5]
    #tr.fill = 'tozeroy'           # drop the filled polygon (saves ~50%)
    tr.line.width = 2       # optional: thinner lines


# Show the figure
fig.show()


In [ ]:
sites = site  # Total number of sites, replace `site` with actual variable if not defined
initial_site_index = 985

# Creating the initial figure that will be updated
fig = go.Figure()

# Loop to add all KDE plots to the figure but set them to be invisible initially
for idx in range(sites):
    data = [theta_unadjusted[:, idx], theta_adjusted_b0[:, idx], theta_adjusted_b15[:, idx]]
    group_labels = ['Baseline', 'b=0', 'b=15']
    colors = ['black', 'red', 'blue']
    
    # We create one distplot for each site and store them in different traces
    distplot = ff.create_distplot(data, group_labels, show_hist=False, show_rug=False, colors=colors,bin_size=1)
    trace_idx = 0
    for trace in distplot.data:
        trace.visible = False  # Set all to invisible initially
        if trace_idx == 1 or trace_idx == 2 :
            trace.update(mode='lines', fill='tozeroy')
        trace.update(line=dict(width=4))
        fig.add_trace(trace)
        trace_idx += 1

# Creating buttons for the dropdown menu
buttons = []
initial_visibility = [False] * len(fig.data) 
for idx in range(sites):
    visibility = [False] * len(fig.data)
    
    # We need to toggle visibility of traces in groups of 3 (as each distplot added 3 traces)
    for j in range(idx * 3, (idx + 1) * 3):
        visibility[j] = True
    
    button = dict(
        label=f'Site {idx + 1}',
        method='update',
        args=[{'visible': visibility},
              {'title': f'Figure 9(b) Probability Density for θ at Site {idx + 1}'}]
    )
    buttons.append(button)
    if idx == initial_site_index:
        initial_visibility = visibility.copy()
        
for i, vis in enumerate(initial_visibility):
    fig.data[i].visible = vis        
# Adding the dropdown menus to the figure
fig.update_layout(
    updatemenus=[{
        'buttons': buttons,
        'active': initial_site_index,
        'direction': 'down',
        'showactive': True,
        'x': 0.8,
        'xanchor': 'center',
        'y': 1.15,
        'yanchor': 'top'
    }],
    title=f'Probability Density for θ at Site {initial_site_index + 1}',
    xaxis_title='Parameter Value',
    yaxis_title='Density',
    width=800,  # Width of the figure in pixels
    height=600  # Height of the figure in pixels
)

for tr in fig.data:
    tr.x = tr.x[::5]        # 300 pts → 60 pts
    tr.y = tr.y[::5]
    #tr.fill = 'tozeroy'          # drop the filled polygon (saves ~50%)
    tr.line.width = 2       # optional: thinner lines


# Show the figure
fig.show()


### Value decomposition

We then compute the present values under ambiguity aversion in comparison to ambiguity neutrality, for different levels of $\xi$.

In [ ]:
import pandas as pd
import numpy as np
import os
import ipywidgets as widgets
from IPython.display import display, HTML
from IPython.display import display, Math, Latex
root_folder='/project/lhansen/HMC_book/Amazon/docs'
os.chdir(root_folder)

def rename(df):
    df.columns=['Pa','Pe','b','agricultural output','net transfers','forest services','adjustment costs','planner value']
    # Define the custom values for the first row
    custom_values = ['($)'] + ['($)'] + ['($)'] + ['($10^11)'] * (len(df.columns) - 3)
    
    # Create a new DataFrame for the new row with the custom values
    new_row = pd.DataFrame([custom_values], columns=df.columns)
    
    # Concatenate this new row to the top of the existing DataFrame
    df = pd.concat([new_row, df], ignore_index=True)
    
    return df
# Example DataFrames for Model A and Model B
df_det_1043 = pd.read_csv(os.getcwd()+'/data/1043site/det/pv_41.11.csv')
df_det_1043 = rename(df_det_1043)
df_det_78 = pd.read_csv(os.getcwd()+'/data/78site/det/pv_41.11.csv')
df_det_78 = rename(df_det_78)
df_xi1 = pd.read_csv(os.getcwd()+'/data/1043site/hmc/pv_xi1.csv')
#df_xi1 = rename(df_xi1)
df_xi2 = pd.read_csv(os.getcwd()+'/data/1043site/hmc/pv_xi2.csv')
#df_xi2 = rename(df_xi2)
df_xi05 = pd.read_csv(os.getcwd()+'/data/1043site/hmc/pv_xi05.csv')
#df_xi05 = rename(df_xi05)
df_mpc=pd.read_csv(os.getcwd()+'/data/78site/mpc/mpc.csv',na_filter=False)


def create_df_widget(df,subtitle,tag_id):
    """Utility function to create a widget for displaying a DataFrame with centered cells."""
    # Define CSS to center text in table cells
    style = """
    <style>
        .dataframe td, .dataframe th {
            text-align: center;
            vertical-align: middle;
        }
        .dataframe thead th {
            background-color: #f2f2f2;  # Light gray background in the header
        }
    </style>
    """
    # Convert DataFrame to HTML and manually add the 'id' attribute
    html_df = df.to_html(index=False)
    html_df = html_df.replace('<table border="1" class="dataframe">', f'<table id="{tag_id}" border="1" class="dataframe">')

    html = style + html_df
    html_widget = widgets.HTML(value=html)  # Use ipywidgets.HTML here
    subtitle_widget = widgets.Label(value=subtitle, layout=widgets.Layout(justify_content='center'))
    out = widgets.VBox([subtitle_widget, html_widget], layout={'border': '1px solid black'})
    return out



# Tab widget to hold different models
tab = widgets.Tab()
children = [create_df_widget(df_xi2,'Present-value decomposition - parameter ambiguity','tab:valueObjectiveDecomposition_1043sites_det3'),
            create_df_widget(df_xi1,'Present-value decomposition - parameter ambiguity','tab:valueObjectiveDecomposition_1043sites_det4'),
            create_df_widget(df_xi05,'Present-value decomposition - parameter ambiguity','tab:valueObjectiveDecomposition_1043sites_det4'),
            ]
tab.children = children
for i, title in enumerate(['1043site ξ = 2 ','1043site ξ = 1 ', '1043site ξ = 0.5 ']):
    tab.set_title(i, title)

# Display the tab widget
tab.selected_index = 0
display(tab)


<br>
<hr style="height:4px; background-color:black; border:none;">